# 📋 Python Lists & Arrays — The Master Guide
*From Zero to Interview-Ready*

---

## Mental Model

Think of a Python list as a **hallway of numbered boxes** — each box has a direct address (index). You can jump straight to any box in O(1) time.

When the hallway runs out of room, Python **rents a 2× bigger hallway** and moves everything over. This is why `append` is O(1) *amortized* — the occasional move is expensive, but it happens so rarely that the average cost per append is constant.

Adding or removing from the **middle** forces every box behind the insertion point to shuffle one step — that is the O(n) cost of `insert(i, x)` and `pop(i)`.

---

## Table of Contents

| # | Section | Link |
|---|---------|------|
| 1 | What Is a List? The Visual Model | [→ go](#1) |
| 2 | Creating / Setup | [→ go](#2) |
| 3 | The Core API — All Operations | [→ go](#3) |
| 4 | Decision Map — When To Use What | [→ go](#4) |
| 5 | Pattern 1: List Internals — dynamic array, append, insert, pop (LC 238) | [→ go](#5) |
| 6 | Pattern 2: Copy Trap — assignment vs slice vs copy (LC 48) | [→ go](#6) |
| 7 | Pattern 3: 2D List Initialization — aliased rows trap vs correct (LC 74) | [→ go](#7) |
| 8 | Pattern 4: Slicing and Negative Indexing — windows, reversal, step (LC 344) | [→ go](#8) |
| 9 | Pattern 5: In-place vs Copy — sort vs sorted, reverse vs reversed (LC 75) | [→ go](#9) |
| 10 | The Lists & Arrays Decision Map | [→ go](#10) |
| 11 | Interview Cheat Sheet | [→ go](#11) |

<a id='1'></a>

## 1. What Is a List? The Visual Model

```
                    PYTHON LIST — THE HALLWAY OF BOXES

  index:    0       1       2       3       4
           ┌───────┬───────┬───────┬───────┬───────┐  ← contiguous memory
           │  10   │  20   │  30   │  40   │  50   │
           └───────┴───────┴───────┴───────┴───────┘
                                                   ↑
                                        len=5, capacity may be 8

  Negative:  -5      -4      -3      -2      -1
  (same boxes — counted from the right instead of the left)

  OPERATION         COST          WHY
  ─────────────────────────────────────────────────────────────────
  a[i]              O(1)          direct address — jump straight to box i
  a[-1]             O(1)          direct address — same mechanism
  a.append(x)       O(1) amort    add to right end — no shifting needed
  a.pop()           O(1)          remove from right end — no shifting needed
  a.insert(0, x)    O(n)          every box slides one step right to make room
  a.pop(0)          O(n)          every box slides one step left to fill gap
  a[i:j]            O(k)          copies k=j-i boxes into a new hallway
  a[:]              O(n)          copies every box into a new hallway

  WHY DOUBLING MATTERS
  ────────────────────
  Hallway full → Python rents 2× bigger → moves n boxes (one O(n) event).
  Doubling happens at n, 2n, 4n... so total moves across n appends:
  n + n/2 + n/4 + ... = 2n → O(1) per append on average.
  This is amortized O(1) — it is real, not magic.
```

<a id='2'></a>

## 2. Creating / Setup

In [ ]:
# ── CREATING LISTS — every form you will actually use ────

empty       = []                                # empty — zero boxes
from_range  = list(range(5))                    # [0, 1, 2, 3, 4]
from_string = list("abc")                       # ['a', 'b', 'c'] — one char per box
filled      = [0] * 5                           # [0, 0, 0, 0, 0] — 5 boxes all zero
literal     = [10, 20, 30, 40, 50]             # explicit values
from_comp   = [x * 2 for x in range(5)]        # [0, 2, 4, 6, 8] — list comprehension

# ── 2D — SAFE initialization ──────────────────────────────
grid_bad  = [[0] * 3] * 3          # TRAP — 3 aliases to same row
grid_good = [[0] * 3 for _ in range(3)]  # 3 independent rows — correct

print(f"empty       : {empty}")
print(f"from_range  : {from_range}")
print(f"from_string : {from_string}")
print(f"filled      : {filled}")
print(f"literal     : {literal}")
print(f"from_comp   : {from_comp}")
print(f"grid_bad    : {grid_bad}")
print(f"grid_good   : {grid_good}")

# prove the trap
grid_bad[0][0]  = 99
grid_good[0][0] = 99
print(f"bad  after [0][0]=99 : {grid_bad}")   # all 3 rows changed — trap!
print(f"good after [0][0]=99 : {grid_good}")  # only row 0 changed — correct

# Simplicity and clarity is Gold

<a id='3'></a>

## 3. The Core API — All Operations

```
OPERATION              COMPLEXITY       WHAT IT DOES
───────────────────────────────────────────────────────────────────
a[i]                   O(1)             read or write element at index i
a[-1]                  O(1)             last element — counts from right
a.append(x)            O(1) amortized   add x to the right end
a.pop()                O(1)             remove and return last element
a.pop(i)               O(n)             remove at index i, shift left
a.insert(i, x)         O(n)             insert at index i, shift right
a[i:j]                 O(k)             copy slice, k = j-i elements
a[i:j:step]            O(k)             copy slice with step — a[::-1] reverses
a[:]                   O(n)             full shallow copy
a.copy()               O(n)             same as a[:]
len(a)                 O(1)             number of elements
a.sort()               O(n log n)       in-place sort — mutates a — returns None
sorted(a)              O(n log n)       returns new sorted list — a untouched
a.reverse()            O(n)             in-place reverse — mutates a — returns None
reversed(a)            O(n)             returns iterator — a untouched
x in a                 O(n)             linear scan — use set for O(1)
a.extend(b)            O(k)             append all k elements of b to a
a + b                  O(n+m)           new list — a and b both unchanged
a.index(x)             O(n)             index of first x — raises ValueError if absent
a.count(x)             O(n)             how many times x appears

THINGS YOU DO NOT DO
─────────────────────────────────────────────────────────────────
❌  b = a              b is an alias — b[0] = 99 also changes a
❌  [[0]*3] * 3        all rows alias the same row — change one, change all
❌  a.insert(0, x) in a loop   O(n) per call — use collections.deque instead
❌  x in a in a tight loop     O(n) per lookup — convert to set first
❌  row = row[::-1]    creates new list — if row is inside matrix, matrix is unchanged
```

In [ ]:
# ── CORE API DEMO — run this cell, read every print ──────

a = [10, 20, 30, 40, 50]
print(f"start             : {a}")

# random access — O(1)
print(f"a[0]              : {a[0]}")       # 10
print(f"a[-1]             : {a[-1]}")      # 50
print(f"a[1:3]            : {a[1:3]}")     # [20, 30]
print(f"a[::-1]           : {a[::-1]}")    # [50, 40, 30, 20, 10]

# append / pop right — O(1)
a.append(60)
print(f"after append(60)  : {a}")
a.pop()
print(f"after pop()       : {a}")

# insert / pop at position — O(n)
a.insert(0, 0)
print(f"after insert(0,0) : {a}")
a.pop(0)
print(f"after pop(0)      : {a}")

# sort — in-place mutates vs sorted makes new list
b = sorted(a, reverse=True)
print(f"sorted(a) result  : {b}")          # new list
print(f"a still            : {a}")         # unchanged
a.sort()
print(f"a.sort()           : {a}")         # mutated

# reverse — in-place mutates vs reversed returns iterator
a.reverse()
print(f"a.reverse()        : {a}")         # mutated in-place
print(f"reversed(a) list   : {list(reversed(a))}")  # new list — a untouched

# Simplicity and clarity is Gold

<a id='4'></a>

## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                    WHAT TO DO
───────────────────────────────────────────────────────────────────────
need random access by index              list — O(1) read/write
only appending to the right end          list.append — O(1) amortized
need fast insert/remove at BOTH ends     collections.deque
building a 2D grid                       list comprehension — NOT * operator
need a safe copy you can mutate          a[:] or a.copy() — NOT b = a
reversing the whole list                 a.reverse() in-place, or a[::-1] copy
windowing / subarray                     a[i:j] — O(k) copy
sorting without mutating original        sorted(a) — returns new list
sorting in-place, no extra memory        a.sort() — returns None
want last element fast                   a[-1] — O(1)
want to remove last element fast         a.pop() — O(1)
want to insert or remove at front        use deque — not a.insert(0,x)
```

<a id='5'></a>

## 5. 🧩 Pattern 1: List Internals — dynamic array, append, insert, pop — LC 238

```
PROBLEM:
  LC 238 — Product of Array Except Self
  Given nums, return output where output[i] = product of all elements
  except nums[i]. No division allowed. O(n) time.

APPROACH:
  Two passes over a single output list.
  Left pass: output[i] = product of everything to the LEFT of i.
  Right pass: multiply output[i] by product of everything to the RIGHT of i.
  A single running variable tracks the right product — no extra array needed.

SLOW MOTION TRACE on nums = [1, 2, 3, 4]:
  ── left pass (prefix) ──
  i=0: output=[1, ?, ?, ?]   prefix was 1, now prefix = 1*1 = 1
  i=1: output=[1, 1, ?, ?]   prefix was 1, now prefix = 1*2 = 2
  i=2: output=[1, 1, 2, ?]   prefix was 2, now prefix = 2*3 = 6
  i=3: output=[1, 1, 2, 6]   prefix was 6

  ── right pass (suffix) ──
  i=3: output[3] = 6*1  = 6,   suffix = 1*4 = 4
  i=2: output[2] = 2*4  = 8,   suffix = 4*3 = 12
  i=1: output[1] = 1*12 = 12,  suffix = 12*2 = 24
  i=0: output[0] = 1*24 = 24

  result: [24, 12, 8, 6]  ✓

KEY INSIGHT:
  The output list is used as a running accumulator — written left to right,
  then multiplied right to left. No division, no extra O(n) space.

TIME / SPACE:
  Time:  O(n) — two linear passes
  Space: O(1) extra — output list excluded from space count per problem rules
```

In [ ]:
def product_except_self(nums: list) -> list:
    """
    LC 238 — Product of Array Except Self
    Approach: two-pass prefix/suffix accumulation on a single output list.
    Args:
        nums (list): integer array, len >= 2, no division allowed.
    Returns:
        list: output[i] = product of all elements except nums[i].
    Time:  O(n) — one left pass, one right pass
    Space: O(1) extra — output list not counted per problem constraint
    """
    n = len(nums)
    output = [1] * n            # pre-fill with 1s — multiplication neutral element

    # ── LEFT PASS — output[i] = product of everything to the LEFT of i ──
    prefix = 1                  # nothing is to the left of index 0 yet
    for i in range(n):
        output[i] = prefix      # record the product of all elements left of i
        prefix *= nums[i]       # extend the left window to include nums[i]

    # ── RIGHT PASS — multiply output[i] by product of everything to the RIGHT of i ──
    # slow motion on nums = [1, 2, 3, 4]:
    # before: output = [1, 1, 2, 6]
    # i=3: output[3] = 6  * 1  = 6,   suffix = 1*4  = 4
    # i=2: output[2] = 2  * 4  = 8,   suffix = 4*3  = 12
    # i=1: output[1] = 1  * 12 = 12,  suffix = 12*2 = 24
    # i=0: output[0] = 1  * 24 = 24
    # result: [24, 12, 8, 6]
    suffix = 1
    for i in range(n - 1, -1, -1):
        output[i] *= suffix     # bake in the product of everything to the right
        suffix *= nums[i]       # extend the right window to include nums[i]

    return output


def test_harness(fn):
    tests = [
        ([1, 2, 3, 4],         [24, 12, 8, 6]),      # standard case
        ([-1, 1, 0, -3, 3],    [0, 0, 9, 0, 0]),     # zero in array
        ([2, 3],                [3, 2]),               # minimal two elements
        ([1, 0],                [0, 1]),               # zero at end
        ([0, 0],                [0, 0]),               # two zeros
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(product_except_self)

print(product_except_self([1, 2, 3, 4]))        # [24, 12, 8, 6]
print(product_except_self([-1, 1, 0, -3, 3]))   # [0, 0, 9, 0, 0]

print("product_except_self defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Copy Trap — assignment vs slice vs copy — LC 48

```
PROBLEM:
  LC 48 — Rotate Image
  Rotate an n×n matrix 90° clockwise. In-place — no extra 2D matrix.

APPROACH:
  Two in-place steps: transpose then reverse each row.
  Transpose: swap matrix[i][j] with matrix[j][i] (flip along diagonal).
  Reverse rows: row.reverse() — each row becomes its own mirror image.
  Together, these two steps equal a 90° clockwise rotation.

COPY TRAP RELEVANCE:
  row.reverse() mutates the row inside the matrix — correct.
  row = row[::-1] creates a NEW list and discards it — matrix unchanged.
  Knowing which operations mutate and which create is the whole game here.

SLOW MOTION TRACE on 3×3:
  input:      transpose:    reverse rows:
  1 2 3       1 4 7         7 4 1
  4 5 6  →    2 5 8    →    8 5 2   ✓ (90° clockwise)
  7 8 9       3 6 9         9 6 3

KEY INSIGHT:
  Any in-place rotation requires the two-step dance: transpose + row-reverse.
  Attempting direct rotation needs a temporary copy — that is O(n²) space.

TIME / SPACE:
  Time:  O(n²) — visit every cell twice
  Space: O(1)  — pure in-place
```

In [ ]:
def rotate(matrix: list) -> None:
    """
    LC 48 — Rotate Image
    Approach: transpose in-place then reverse each row in-place.
    Args:
        matrix (list[list[int]]): n×n matrix, modified in-place.
    Returns:
        None: mutates matrix directly — no return value.
    Time:  O(n²) — two full passes over all n² cells
    Space: O(1)  — no auxiliary matrix allocated
    """
    n = len(matrix)

    # ── STEP 1: TRANSPOSE — swap upper and lower triangles ──
    # only visit upper triangle (j starts at i+1) — avoids double-swapping
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]

    # ── STEP 2: REVERSE EACH ROW — in-place ──
    # THE COPY TRAP LIVES HERE:
    #   row.reverse()      ← mutates the row that is inside matrix — CORRECT
    #   row = row[::-1]    ← creates a new list, rebinds local var — matrix UNCHANGED
    for row in matrix:
        row.reverse()


def test_harness(fn):
    import copy
    tests = [
        ([[1,2,3],[4,5,6],[7,8,9]],
         [[7,4,1],[8,5,2],[9,6,3]]),
        ([[5,1,9,11],[2,4,8,10],[13,3,6,7],[15,14,12,16]],
         [[15,13,2,5],[14,3,4,1],[12,6,8,9],[16,7,10,11]]),
        ([[1]],
         [[1]]),
        ([[1,2],[3,4]],
         [[3,1],[4,2]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        matrix = copy.deepcopy(inputs[0])   # rotate mutates in-place — need fresh copy per test
        fn(matrix)
        got = matrix
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(rotate)

m = [[1,2,3],[4,5,6],[7,8,9]]
rotate(m)
print(m)    # [[7,4,1],[8,5,2],[9,6,3]]

print("rotate defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: 2D List Initialization — aliased rows trap vs correct — LC 74

```
PROBLEM:
  LC 74 — Search a 2D Matrix
  Rows are sorted. Last element of row i < first element of row i+1.
  Find target — return True/False.

APPROACH:
  The whole matrix is one sorted list folded into rows.
  Map virtual flat index mid to row = mid // n, col = mid % n.
  Binary search on the virtual 1D array — no copy or flattening needed.

2D INITIALIZATION RELEVANCE:
  [[0]*n]*m creates m aliases to the SAME row — all rows change together.
  [[0]*n for _ in range(m)] creates m independent rows — each can change alone.
  mid // n and mid % n only produce correct results on a properly initialized grid.

SLOW MOTION TRACE on matrix = [[1,3,5,7],[10,11,16,20],[23,30,34,60]], target=3:
  m=3, n=4 → virtual indices 0..11
  lo=0, hi=11

  mid=5  → row=5//4=1, col=5%4=1 → val=matrix[1][1]=11 → 11>3  → hi=4
  mid=2  → row=2//4=0, col=2%4=2 → val=matrix[0][2]=5  → 5>3   → hi=1
  mid=0  → row=0//4=0, col=0%4=0 → val=matrix[0][0]=1  → 1<3   → lo=1
  mid=1  → row=1//4=0, col=1%4=1 → val=matrix[0][1]=3  → found! return True  ✓

KEY INSIGHT:
  row = mid // n, col = mid % n — integer division gives row, modulo gives column.
  This mapping turns any 2D grid into a standard binary search.

TIME / SPACE:
  Time:  O(log(m*n)) — binary search on virtual flat array of size m*n
  Space: O(1)        — no extra data structure
```

In [ ]:
def search_matrix(matrix: list, target: int) -> bool:
    """
    LC 74 — Search a 2D Matrix
    Approach: treat the matrix as a virtual sorted 1D array, binary search it.
    Args:
        matrix (list[list[int]]): m×n matrix, rows sorted, rows sorted relative to each other.
        target (int): value to find.
    Returns:
        bool: True if target exists in matrix.
    Time:  O(log(m*n)) — one binary search over m*n virtual positions
    Space: O(1)        — only index variables
    """
    m = len(matrix)
    n = len(matrix[0])
    lo, hi = 0, m * n - 1      # virtual flat indices for the whole grid

    while lo <= hi:
        mid = (lo + hi) // 2
        row = mid // n          # which row does this virtual index land on?
        col = mid % n           # which column within that row?
        val = matrix[row][col]

        # slow motion on [[1,3,5,7],[10,11,16,20],[23,30,34,60]], target=3:
        # mid=5  row=1 col=1 val=11  11>3  hi=4
        # mid=2  row=0 col=2 val=5   5>3   hi=1
        # mid=0  row=0 col=0 val=1   1<3   lo=1
        # mid=1  row=0 col=1 val=3   found!

        if val == target:
            return True
        elif val < target:
            lo = mid + 1        # target is to the right — move window right
        else:
            hi = mid - 1        # target is to the left — move window left

    return False


def test_harness(fn):
    tests = [
        ([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 3,   True),
        ([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 13,  False),
        ([[1]],                                    1,   True),
        ([[1]],                                    2,   False),
        ([[1,3],[5,7]],                            7,   True),
        ([[1,3],[5,7]],                            4,   False),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(search_matrix)

print(search_matrix([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 3))   # True
print(search_matrix([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 13))  # False

print("search_matrix defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: Slicing and Negative Indexing — windows, reversal, step — LC 344

```
PROBLEM:
  LC 344 — Reverse String
  Reverse a list of characters in-place. O(1) extra space.

APPROACH:
  Two-pointer swap: left starts at 0, right starts at -1 (last index).
  Each step swaps s[left] and s[right], then close the gap.
  Stop when the pointers meet in the middle.

SLICING RELEVANCE:
  s[::-1] would reverse but creates a COPY — O(n) space, not in-place.
  s.reverse() works in-place but is O(n) — this is the "correct" solution for LC.
  The two-pointer version is the manual version that shows the mechanism.

NEGATIVE INDEXING REFERENCE:
  a[-1]    → last element        (same as a[len(a)-1])
  a[-2]    → second to last
  a[-n:]   → last n elements     (tail window)
  a[:-n]   → everything except last n
  a[::-1]  → full reverse copy   (step = -1)
  a[i:j:2] → every other element from i to j

SLOW MOTION TRACE on s = ['h','e','l','l','o']:
  left=0, right=4: swap s[0]↔s[4] → ['o','e','l','l','h']
  left=1, right=3: swap s[1]↔s[3] → ['o','l','l','e','h']
  left=2, right=2: left >= right → stop
  result: ['o','l','l','e','h']  ✓

KEY INSIGHT:
  The two pointers walk toward each other — each swap places both characters
  in their final position. Stops at the midpoint, not at the end.

TIME / SPACE:
  Time:  O(n) — n/2 swaps
  Space: O(1) — in-place, no copy
```

In [ ]:
def reverse_string(s: list) -> None:
    """
    LC 344 — Reverse String
    Approach: two-pointer swap walking inward from both ends.
    Args:
        s (list[str]): list of characters, modified in-place.
    Returns:
        None: mutates s directly.
    Time:  O(n) — n/2 swaps, one per pair
    Space: O(1) — no extra list allocated
    """
    left, right = 0, len(s) - 1    # left door and right door of the hallway

    while left < right:
        # slow motion on ['h','e','l','l','o']:
        # step 1: left=0 right=4  swap h↔o  → ['o','e','l','l','h']
        # step 2: left=1 right=3  swap e↔l  → ['o','l','l','e','h']
        # step 3: left=2 right=2  stop — pointers met
        s[left], s[right] = s[right], s[left]   # swap the two boxes
        left  += 1                              # close from the left
        right -= 1                              # close from the right


def test_harness(fn):
    tests = [
        (['h','e','l','l','o'],    ['o','l','l','e','h']),
        (['H','a','n','n','a','h'],['h','a','n','n','a','H']),
        (['a'],                    ['a']),
        (['a','b'],                ['b','a']),
    ]
    passed = 0
    for *inputs, expected in tests:
        s = list(inputs[0])     # copy so we can check without mutating the test tuple
        fn(s)
        got = s
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(reverse_string)

# bonus — slicing reference
nums = [10, 20, 30, 40, 50]
print(f"last       : {nums[-1]}")         # 50
print(f"second last: {nums[-2]}")         # 40
print(f"last 3     : {nums[-3:]}")        # [30, 40, 50]
print(f"all but 2  : {nums[:-2]}")        # [10, 20, 30]
print(f"reversed   : {nums[::-1]}")       # [50, 40, 30, 20, 10]  — copy
print(f"every 2nd  : {nums[::2]}")        # [10, 30, 50]
print(f"middle     : {nums[1:4]}")        # [20, 30, 40]

print("reverse_string defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: In-place vs Copy — sort vs sorted, reverse vs reversed — LC 75

```
PROBLEM:
  LC 75 — Sort Colors (Dutch National Flag)
  Sort array of 0s, 1s, 2s in-place in one pass. No sort() call.

APPROACH:
  Three-pointer partition: lo (left boundary of 0s), mid (cursor), hi (right boundary of 2s).
  Loop while mid <= hi:
    nums[mid]==0 → swap with lo, advance both lo and mid
    nums[mid]==1 → mid is in the right zone, just advance mid
    nums[mid]==2 → swap with hi, retreat hi (do NOT advance mid — newly swapped value unverified)

IN-PLACE VS COPY RELEVANCE:
  sort() / reverse() → mutate in-place, return None
  sorted() / reversed() → return new object, original unchanged
  This problem forces in-place because no allocation of extra memory is allowed.

  MUTATION                   COPY
  ─────────────────────────────────────────────────────
  a.sort()         O(n log n)    sorted(a)     O(n log n)
  a.reverse()      O(n)          a[::-1]       O(n)
  a[i],a[j]=...    O(1) swap     (a[j],a[i])   O(1) new tuple (temp)

SLOW MOTION TRACE on nums = [2, 0, 2, 1, 1, 0]:
  lo=0 mid=0 hi=5
  nums[0]=2 → swap(mid,hi): [0,0,2,1,1,2] hi=4
  nums[0]=0 → swap(lo,mid): [0,0,2,1,1,2] lo=1 mid=1
  nums[1]=0 → swap(lo,mid): [0,0,2,1,1,2] lo=2 mid=2
  nums[2]=2 → swap(mid,hi): [0,0,1,1,2,2] hi=3
  nums[2]=1 → mid=3
  nums[3]=1 → mid=4
  mid=4 > hi=3 → stop
  result: [0,0,1,1,2,2]  ✓

KEY INSIGHT:
  lo..mid-1 = confirmed 0s zone. mid..hi = unknown zone. hi+1..end = confirmed 2s.
  When you swap a 2 from mid to hi, the value that arrives at mid is unknown — do not advance mid.

TIME / SPACE:
  Time:  O(n) — single pass
  Space: O(1) — three index variables
```

In [ ]:
def sort_colors(nums: list) -> None:
    """
    LC 75 — Sort Colors (Dutch National Flag)
    Approach: three-pointer in-place partition in a single pass.
    Args:
        nums (list[int]): array of 0s, 1s, 2s — modified in-place.
    Returns:
        None: mutates nums directly.
    Time:  O(n) — single pass, each element visited at most twice
    Space: O(1) — three index variables only
    """
    lo  = 0             # left boundary — everything left of lo is a confirmed 0
    mid = 0             # cursor — everything left of mid is confirmed (0 or 1)
    hi  = len(nums) - 1 # right boundary — everything right of hi is a confirmed 2

    while mid <= hi:
        if nums[mid] == 0:
            # slow motion on [2,0,2,1,1,0]:
            # mid=0 val=2 → swap(0,5) → [0,0,2,1,1,2] hi=4
            # mid=0 val=0 → swap(0,0) → lo=1 mid=1
            # mid=1 val=0 → swap(1,1) → lo=2 mid=2
            # mid=2 val=2 → swap(2,4) → [0,0,1,1,2,2] hi=3
            # mid=2 val=1 → mid=3
            # mid=3 val=1 → mid=4
            # mid=4 > hi=3 → done
            nums[lo], nums[mid] = nums[mid], nums[lo]   # send 0 to the confirmed-0 zone
            lo  += 1    # left boundary expands right
            mid += 1    # cursor advances — the value we placed at lo was already confirmed
        elif nums[mid] == 1:
            mid += 1    # 1 belongs exactly here — just advance
        else:
            nums[mid], nums[hi] = nums[hi], nums[mid]   # send 2 to the confirmed-2 zone
            hi -= 1     # right boundary shrinks left
            # do NOT advance mid — the value that arrived at mid from hi is unverified


def test_harness(fn):
    tests = [
        ([2,0,2,1,1,0],  [0,0,1,1,2,2]),
        ([2,0,1],         [0,1,2]),
        ([0],             [0]),
        ([1],             [1]),
        ([2,2,2],         [2,2,2]),
        ([0,0,0],         [0,0,0]),
        ([1,2,0],         [0,1,2]),
    ]
    passed = 0
    for *inputs, expected in tests:
        nums = list(inputs[0])     # copy — sort_colors mutates in-place
        fn(nums)
        got = nums
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(sort_colors)

n = [2, 0, 2, 1, 1, 0]
sort_colors(n)
print(n)    # [0, 0, 1, 1, 2, 2]

print("sort_colors defined.")

<a id='10'></a>

## 10. The Lists & Arrays Decision Map

```
QUESTION TYPE                        KEY TECHNIQUE                    LC PROBLEMS
────────────────────────────────────────────────────────────────────────────────────
prefix / suffix product without /    two-pass accumulation            238
in-place matrix rotation             transpose + reverse rows         48
search sorted 2D matrix              binary search with // and %      74, 240
reverse in-place, O(1) space         two-pointer inward swap          344
partition by value in one pass       Dutch National Flag (3 pointers) 75
sort stably without mutating input   sorted(a)                        215
sort in-place (no extra memory)      a.sort() or DNF pattern          75
copy safely before mutating          a[:] or a.copy()                 48, 54
build 2D grid correctly              list comprehension, not * op     48, 74
sliding window / subarray            a[i:j] slice or two pointers     3, 76
```

<a id='11'></a>

## 11. Interview Cheat Sheet

### 1. When to reach for a list

| Signal | What to Do |
|--------|------------|
| Need random access by index | `list` — O(1) read/write |
| Only appending to the right | `list.append` — O(1) amortized |
| Fast insert/remove at both ends | `collections.deque` |
| Building a 2D grid | List comprehension — NOT `*` operator |
| Need a safe copy to mutate | `a[:]` or `a.copy()` — NOT `b = a` |
| Sorting without touching original | `sorted(a)` — returns new list |
| Sorting in-place, no extra memory | `a.sort()` — returns None |
| Membership check in a tight loop | Convert to `set` first — O(1) lookup |

### 2. The O(1) operations — memorize these

```python
a[i]        # read or write any index — O(1)
a[-1]       # last element — O(1)
a.append(x) # add to right end — O(1) amortized
a.pop()     # remove last element — O(1)
len(a)      # count of elements — O(1)
```

### 3. Common templates

```python
# PREFIX / SUFFIX PRODUCT  (LC 238 pattern)
n = len(nums)
output = [1] * n
prefix = 1
for i in range(n):
    output[i] = prefix
    prefix *= nums[i]
suffix = 1
for i in range(n - 1, -1, -1):
    output[i] *= suffix
    suffix *= nums[i]

# TWO-POINTER INWARD SWAP  (LC 344 pattern)
left, right = 0, len(s) - 1
while left < right:
    s[left], s[right] = s[right], s[left]
    left += 1
    right -= 1

# DUTCH NATIONAL FLAG — 3 PARTITIONS  (LC 75 pattern)
lo, mid, hi = 0, 0, len(nums) - 1
while mid <= hi:
    if nums[mid] == 0:
        nums[lo], nums[mid] = nums[mid], nums[lo]
        lo += 1
        mid += 1
    elif nums[mid] == 1:
        mid += 1
    else:
        nums[mid], nums[hi] = nums[hi], nums[mid]
        hi -= 1   # do NOT advance mid

# SAFE 2D GRID
grid = [[0] * n for _ in range(m)]   # m independent rows
# NOT: grid = [[0] * n] * m          # all m rows are the same object

# IN-PLACE vs COPY REMINDER
a.sort()           # mutates a — returns None
b = sorted(a)      # new sorted list — a untouched
a.reverse()        # mutates a — returns None
b = a[::-1]        # new reversed list — a untouched
row.reverse()      # mutates the row inside the matrix — correct
row = row[::-1]    # rebinds local variable only — matrix unchanged
```

### 4. Gotchas

```
❌  b = a                   alias — not a copy, b[0]=99 mutates a
❌  [[0]*n] * m             all m rows are the same object
❌  row = row[::-1]         rebinds local variable — matrix unchanged
❌  a.insert(0, x) in loop  O(n) per call — use deque for front inserts
❌  a.sort() returns None   do not do b = a.sort()
✅  a[:] or a.copy()        safe shallow copy
✅  row.reverse()           mutates in-place — row inside matrix changes
✅  [[0]*n for _ in range(m)]  correct 2D init — independent rows
✅  sorted(a) — a untouched, returns new list
✅  mid // n, mid % n — converts flat index to row, col
```

## 12. Master Map — All 5 Patterns at a Glance

```
              📋 PYTHON LIST — MASTER MAP

              dynamic array in contiguous memory
                           │
          ┌────────────────┼────────────────────────────┐
          │                │                            │
   INTERNALS           COPY RULES                  2D GRIDS
   append O(1)         b = a → alias               [[0]*n for _ in range(m)]
   pop()  O(1)         a[:]  → copy                NOT [[0]*n] * m
   insert O(n)         a.copy() → copy             row // n, col % n
   pop(0) O(n)         row.reverse() mutates       binary search on flat index
          │            row[::-1] copies                     │
          │                    │                            │
     LC 238               LC 48 rotate              LC 74 search 2D
     prefix/suffix         transpose+reverse        LC 240 search II
          │
     SLICING               IN-PLACE vs COPY
     a[i:j]    O(k)        a.sort()    mutates — returns None
     a[::-1]   O(n)        sorted(a)   new list — a untouched
     a[-1]     O(1)        a.reverse() mutates — returns None
     a[-n:]    O(n)        a[::-1]     new list — a untouched
          │                       │
     LC 344 reverse         LC 75 sort colors (DNF)
     LC 125 palindrome      LC 215 kth largest
```

---
*End of Lists & Arrays Master Guide — Sean Edition*